# GP Model Comparison Notebook

Purpose: compare GP families on a common engineering-style benchmark and rank by calibration + runtime.


## Learning Roadmap

- Compare GP families under one consistent benchmark protocol.
- Inspect trade-offs among accuracy, calibration, and runtime.
- Use plots + table to choose deployment candidates.


In [ ]:
# Step 1: import candidate models and benchmark dependencies
# Configure Python path for local package imports
import sys
from pathlib import Path

ROOT = Path.cwd().resolve().parents[1] if Path.cwd().name == 'gp' else Path.cwd().resolve()
if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))

import math
import time
import numpy as np
import torch
import matplotlib.pyplot as plt

plt.style.use('seaborn-v0_8-whitegrid')
torch.manual_seed(42)
np.random.seed(42)

from deepuq.models import (
    DeepKernelGaussianProcessRegressor,
    GaussianProcessRegressor,
    HeteroscedasticGaussianProcessRegressor,
    MultiTaskGaussianProcessRegressor,
    RBFKernel,
    SparseGaussianProcessRegressor,
    SpectralMixtureGaussianProcessRegressor,
)


In [ ]:
# Step 2: define shared regression metrics
def regression_metrics(y_true, mean, var):
    y_true = y_true.reshape(-1)
    mean = mean.reshape(-1)
    var = var.reshape(-1).clamp_min(1e-8)
    rmse = torch.sqrt(torch.mean((mean - y_true) ** 2)).item()
    nll = 0.5 * torch.mean(torch.log(2 * torch.pi * var) + (y_true - mean) ** 2 / var).item()
    std = torch.sqrt(var)
    lower = mean - 1.96 * std
    upper = mean + 1.96 * std
    coverage95 = torch.mean(((y_true >= lower) & (y_true <= upper)).float()).item()
    width95 = torch.mean(upper - lower).item()
    return {
        'rmse': rmse,
        'nll': nll,
        'coverage95': coverage95,
        'interval_width95': width95,
    }


In [ ]:
# Step 3: generate benchmark dataset
x_train = torch.linspace(-3.5, 3.5, 120).unsqueeze(-1)
true_fn = lambda x: 0.2 * x + torch.sin(1.4 * x) + 0.2 * torch.sin(4.0 * x)
y_train = true_fn(x_train) + (0.07 + 0.03 * torch.abs(x_train)) * torch.randn_like(x_train)

x_test = torch.linspace(-5.5, 5.5, 300).unsqueeze(-1)
y_test = true_fn(x_test)


In [ ]:
# Step 4: train/evaluate each model and collect metrics
results = []

candidates = {
    'exact_gp': lambda: GaussianProcessRegressor(kernel=RBFKernel(lengthscale=0.8, outputscale=1.1), noise=0.01),
    'sparse_gp': lambda: SparseGaussianProcessRegressor(num_inducing=40, num_iterations=120, kernel=RBFKernel(lengthscale=0.8, outputscale=1.1)),
    'heterosced_gp': lambda: HeteroscedasticGaussianProcessRegressor(),
    'spectral_mixture_gp': lambda: SpectralMixtureGaussianProcessRegressor(num_mixtures=3, opt_steps=140),
    'deep_kernel_gp': lambda: DeepKernelGaussianProcessRegressor(epochs=120),
}

for name, ctor in candidates.items():
    model = ctor()
    t0 = time.perf_counter()
    model.fit(x_train, y_train)
    train_t = time.perf_counter() - t0
    t1 = time.perf_counter()
    uq = model.predict_uq(x_test)
    infer_t = time.perf_counter() - t1
    m = regression_metrics(y_test.squeeze(-1), uq.mean, uq.total_var)
    results.append({'method': name, 'train_s': train_t, 'infer_s': infer_t, **m})

# Multi-task variant on derived target (deflection + gradient)
deflection = y_train
slope = torch.gradient(y_train.squeeze(-1), spacing=(x_train.squeeze(-1),))[0].unsqueeze(-1)
mt_y = torch.cat([deflection, slope], dim=1)
mt_model = MultiTaskGaussianProcessRegressor(num_tasks=2, kernel=RBFKernel(lengthscale=0.8, outputscale=1.0), opt_steps=120)
mt_model.fit(x_train, mt_y)
mt_uq = mt_model.predict_uq(x_test)
results.append({
    'method': 'multitask_icm_gp(task0)',
    **regression_metrics(y_test.squeeze(-1), mt_uq.mean[:, 0], mt_uq.total_var[:, 0]),
    'train_s': float('nan'),
    'infer_s': float('nan'),
})


In [ ]:
# Step 5: summarize metrics in a sorted table
import pandas as pd

df = pd.DataFrame(results)
display(df.sort_values('nll'))


In [ ]:
# Additional diagnostic: metric and runtime bar charts
# Plotting the table values makes method trade-offs easier to compare visually.
plot_df = df.copy()

fig, axes = plt.subplots(2, 2, figsize=(14, 8))
axes[0, 0].bar(plot_df['method'], plot_df['rmse'])
axes[0, 0].set_title('RMSE (lower better)')
axes[0, 0].tick_params(axis='x', rotation=30)

axes[0, 1].bar(plot_df['method'], plot_df['nll'])
axes[0, 1].set_title('NLL (lower better)')
axes[0, 1].tick_params(axis='x', rotation=30)

axes[1, 0].bar(plot_df['method'], plot_df['coverage95'])
axes[1, 0].axhline(0.95, color='k', ls='--', lw=1)
axes[1, 0].set_title('95% coverage (target ~0.95)')
axes[1, 0].tick_params(axis='x', rotation=30)

runtime = plot_df[['method', 'train_s', 'infer_s']].fillna(0.0)
axes[1, 1].bar(runtime['method'], runtime['train_s'], label='train time (s)')
axes[1, 1].bar(runtime['method'], runtime['infer_s'], bottom=runtime['train_s'], label='inference time (s)')
axes[1, 1].set_title('Runtime profile')
axes[1, 1].tick_params(axis='x', rotation=30)
axes[1, 1].legend(loc='best')

plt.tight_layout()
plt.show()
